## TUGAS MANDIRI 
---

### Konteks / Skenario

Tim engineering platform e-commerce (skenario yang sama dari Pertemuan 2-3) resmi meminta seluruh proses analisis data yang sebelumnya memakai pandas **dipindahkan ke PySpark**, karena volume data transaksi diperkirakan akan tumbuh sangat besar dalam waktu dekat sehingga pandas (yang memuat semua data ke RAM) tidak lagi memadai. Sebagai data analyst yang baru belajar PySpark, anda ditugaskan membuktikan bahwa seluruh alur analisis dapat direplikasi menggunakan PySpark, **membaca data langsung dari HDFS**.

### Menyiapkan Dataset

Jalankan cell untuk membuat dataset baru (transaksi bulan September 2026) dan mengunggahnya ke HDFS.

In [13]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


### Instruksi Pengerjaan

**Seluruhnya wajib menggunakan PySpark, bukan pandas**, dan data **wajib dibaca langsung dari HDFS** (`hdfs://localhost:9000/...`), bukan dari berkas lokal.

---

In [25]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Latihan4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")

SparkSession berhasil dibuat!


---
**A. Membaca dan Eksplorasi Awal** *(bobot 15%)*

Baca dataset dari HDFS, tampilkan `printSchema()`, jumlah baris (`count()`), dan 10 baris pertama (`show(10)`).

In [26]:
#Membaca dataset dari HDFS
df = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True, inferSchema=True
)

print("(1) Tampilan printSchema():")
df.printSchema()

print("(2) Jumlah baris dari HDFS:", df.count())

print("(3) Menampilkan 10  baris pertama:")
df.show(10)

(1) Tampilan printSchema():
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

(2) Jumlah baris dari HDFS: 1000
(3) Menampilkan 10  baris pertama:
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet| 

---
**B. Menangani Data Kosong** *(bobot 15%)*

Kolom `rating` memiliki nilai kosong. Tampilkan berapa banyak, lalu gunakan `df.na.fill()` atau `df.na.drop()` (pilih salah satu, jelaskan alasannya pada markdown cell) untuk menanganinya.

In [27]:
from pyspark.sql.functions import col

kolom_kosong = df.filter(col("rating").isNull()).count()
print("Jumlah nilai rating kosong:", kolom_kosong)

Jumlah nilai rating kosong: 204


In [28]:
#Menggunakan df.na.fill()
df = df.na.fill({"rating": 4})

kolom_kosong = df.filter(col("rating").isNull()).count()
print("Jumlah nilai rating kosong:", kolom_kosong)

Jumlah nilai rating kosong: 0


**ALASAN :** 
Menangani kolom `rating` yang kosong, saya menggunakan `df.na.fill()` untuk mengisi semua nilai yang semulanya null dengan nilai 4 untuk kolom `rating`.

---
**C. Transformasi Data** *(bobot 20%)*

Tambahkan kolom `total_pendapatan` (`unit_terjual x harga_satuan`), lalu tambahkan kolom `tier_transaksi` yang bernilai `"Besar"` jika `total_pendapatan > 500000`, atau `"Kecil"` jika sebaliknya 

In [29]:
from pyspark.sql.functions import col, when

#Menambahkan kolom total_pendapatan dan tier_transaksi
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
df = df.withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

df.show()

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

---
**D. Analisis dengan GroupBy** *(bobot 30%)*

Jawablah dengan kode PySpark (bukan pandas):
1. Kategori apa yang memiliki `total_pendapatan` tertinggi?
2. Kota mana dengan jumlah transaksi **tier "Besar"** terbanyak?
3. Berapa rata-rata `rating` untuk masing-masing `metode_pembayaran`?

In [31]:
from pyspark.sql.functions import sum as spark_sum, avg

kategori_pendapatan = df.groupBy("kategori").agg(spark_sum("total_pendapatan").alias("total_pendapatan")).orderBy(col("total_pendapatan").desc())
kota_tier = df.filter(col("tier_transaksi") == "Besar").groupBy("kota").count().withColumnRenamed("count", "jumlah_transaksi").orderBy(col("jumlah_transaksi").desc())
rating_pembayaran = df.groupBy("metode_pembayaran").agg(avg("rating").alias("avg rating")).orderBy(col("avg rating").desc())

print("(1) Menampilkan kategori dengan total_pendapatan tertinggi:")
kategori_pendapatan.show(1)

print("(2) Menampilkan kota dengan jumlah transaksi tier 'Besar' terbanyak:")
kota_tier.show(1)

print("(3) Rata-rata rating pada masing-masing metode_pembayaran:")
rating_pembayaran.show()

(1) Menampilkan kategori dengan total_pendapatan tertinggi:
+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row

(2) Menampilkan kota dengan jumlah transaksi tier 'Besar' terbanyak:
+----+----------------+
|kota|jumlah_transaksi|
+----+----------------+
|Solo|              92|
+----+----------------+
only showing top 1 row

(3) Rata-rata rating pada masing-masing metode_pembayaran:
+-----------------+-----------------+
|metode_pembayaran|       avg rating|
+-----------------+-----------------+
|              COD|4.139442231075697|
|    Transfer Bank|4.130434782608695|
|         E-Wallet|            4.108|
|     Kartu Kredit|4.085365853658536|
+-----------------+-----------------+



---
**E. Menyimpan Hasil ke HDFS** *(bobot 20%)*

Simpan DataFrame hasil olahan bagian C (lengkap dengan kolom `total_pendapatan` dan `tier_transaksi`) ke HDFS dalam format CSV baru, kemudian verifikasi apakah sudah berhasil.

In [34]:
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/pembaruan_transaksi"

df.write.mode("overwrite").option("header", True).csv(output_path)

print("Hasil berhasil disimpan ke HDFS:")
!hdfs dfs -ls /user/mahasiswa/tugas4/pembaruan_transaksi

Hasil berhasil disimpan ke HDFS:
Found 2 items
-rw-r--r--   3 jauharaputri supergroup          0 2026-09-14 14:36 /user/mahasiswa/tugas4/pembaruan_transaksi/_SUCCESS
-rw-r--r--   3 jauharaputri supergroup      92296 2026-09-14 14:36 /user/mahasiswa/tugas4/pembaruan_transaksi/part-00000-06f7e49d-3a36-4e2f-9a51-60b6e5946539-c000.csv


> **Catatan:** Spark menyimpan hasil sebagai **beberapa berkas partisi** (`part-00000...`, dst.), bukan satu berkas tunggal seperti pandas — ini normal dan justru mencerminkan sifat terdistribusi Spark. Jelaskan secara singkat pada markdown cell mengapa hal ini terjadi

> **Penjelasan:** kita mendapatkan beberapa file dalam satu folder karena Spark menulis setiap partisi data secara langsung ke file `part...` untuk menghindari I/O jaringan.


---

## Menutup SparkSession

Kebiasaan baik: selalu tutup SparkSession di akhir notebook untuk clear up resources (memori & CPU) yang dipakainya.

In [35]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
